# daten einlesen

In [ ]:
season='off' 
#choose between 
#'off' for a total annual view       or
#dry for dry season analysis         or
#wet for wet season analysis

dataset='CMF_low.nc' 

#choose between: 
# 'COT_liq.nc' or    (Cloud Optical Thickness of liqiud clouds)
# 'CMF_low.nc' or    (Cloud Mask Fraction of low clouds) 
# 'CPS_liq.nc'       (Cloud Particle Size of liquid cloud)
#or create a new file with concat.ipynb and choose that one

modis=xr.open_dataset('/projekt1/ag_maahn/data_obs_nobackup/modis/MCD06COSP_D3/merged/'+dataset)

cut_modis=False #true to cut time  e.g. to see dependency of ENSO
start='2005-07-02'  #yyyy-mm-dd start day of modis data, first date is '2002-07-02'
end  ='2024-09-30'  #yyyy-mm-dd last  day of modis data, last  date is '2025-09-30'

if dataset=='COT_liq.nc':
    folder='cot_liq'
    data=folder
    header='Cloud Optical Thickness of liquid clouds' 
    cmin=None
    cmax=None
elif dataset=='CMF_low.nc':
    folder='cmf_low'
    data=folder
    header='Cloud Mask fraction of low clouds'
    cmin=-0.05
    cmax=0.05
elif dataset=='CPS_liq.nc':
    folder='cps_liq'
    data=folder
    header='Cloud Particle Size of liquid clouds'
    cmin=None
    cmax=None

if season=='dry':
    modis = modis.sel(time=modis.time.dt.month.isin([6,7,8,9]))
    header=header+' in dry season'
    data=folder+'_dry'
elif season=='wet':
    modis = modis.sel(time=~modis.time.dt.month.isin([6,7,8,9]))
    header=header+' in wet season'
    data=folder+'_wet'
else:
    pass


if cut_modis:
    modis=modis.sel(time=slice(start, end))

modis['mean_avg']=modis['Mean'].mean(dim='time', skipna=True)

# Trends in Modis daten berechnen

In [ ]:
time_num = (modis.time - modis.time[0]).dt.days.values

def linear_trend_with_pvalue(y):
    mask = ~np.isnan(y)
    if mask.sum() < 2:
        return np.nan, np.nan
    slope, _, _, pvalue, _ = stats.linregress(time_num[mask], y[mask])
    return slope, pvalue

# Slope und p-Wert berechnen
slope, pvalue = xr.apply_ufunc(
    linear_trend_with_pvalue,
    modis["Mean"],
    input_core_dims=[["time"]],
    output_core_dims=[[], []],
    vectorize=True,
    output_dtypes=[float, float]
)

trend_modis = xr.Dataset({
    "trend_Mean": slope.transpose("latitude", "longitude"),
    "pvalue":     pvalue.transpose("latitude", "longitude")
})

trend_modis['trend_Mean']=trend_modis['trend_Mean']*365*10
trend_modis

# PValues plotten

# Trends plotten

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.PlateCarree()})

(trend_modis["pvalue"]).plot.pcolormesh(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    vmin=0,   
    vmax=0.1, 
    cbar_kwargs={"label": "p-wert"}
)

# Non-significant areas hatched (p > 0.05)
#sig_mask = trend_modis["pvalue"] > 0.05
#ax.contourf(
#    trend_modis.longitude, trend_modis.latitude,
#    sig_mask.values,
#    levels=[0.5, 1.5],
#    hatches=["////"],
#    colors="none",
#    transform=ccrs.PlateCarree()
#)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True)
gl.xlocator = mticker.FixedLocator(range(-90, -39, 10))
gl.ylocator = mticker.FixedLocator(range(-10, 11, 10))

ax.set_title(f"Trend {header} 2002–2026\n(Hatching = not significant, p > 0.05)")
#plt.savefig(f'/home/oscholz/plots/{folder}/Trend_{data}.png')
plt.tight_layout()
plt.show()

# In funktionen zusammengefasst:

In [ ]:
def plot(Data, col_min, col_max, colormap="RdBu_r", sig=True , sig_data=trend_modis["pvalue"]):
    
    fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.PlateCarree()})
    
    (Data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap="RdBu_r",
        vmin=col_min,   
        vmax=col_max, 
        cbar_kwargs={"label": "p-wert"}
    )

    if sig and sig_data != data:
    # Non-significant areas hatched (p > 0.05)
        sig_data = trend_modis["pvalue"] > 0.05
        ax.contourf(
            trend_modis.longitude, trend_modis.latitude,
            sig_data.values,
            levels=[0.5, 1.5],
            hatches=["////"],
            colors="none",
            transform=ccrs.PlateCarree()
        )
    
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    
    gl = ax.gridlines(draw_labels=True)
    gl.xlocator = mticker.FixedLocator(range(-90, -39, 10))
    gl.ylocator = mticker.FixedLocator(range(-10, 11, 10))
    
    ax.set_title(f"Trend {header} 2002–2026\n(Hatching = not significant, p > 0.05)")
    plt.savefig(f'/home/oscholz/plots/{folder}/Trend_{data}.png')
    plt.tight_layout()
    plt.show()

In [ ]:
modis = xr.Dataset({
    "Mean"              : Mean.transpose("latitude", "longitude"),
    "Standard_Deviation": Standard_Deviation.transpose("latitude", "longitude"),
    "sum"               : sim.transpose("latitude", "longitude"),
    "Pixel_Counts"      : Pixel_Counts.transpose("latitude", "longitude"),
    "Sum_Squares"       : Sum_Squares.transpose("latitude", "longitude"),
})

# Back Up

In [ ]:
#mean
fig, ax = plt.subplots(figsize=(12, 8),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

modis['mean_avg'].plot.pcolormesh(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="turbo",
    #vmin=cmin,   
    #vmax=cmax, 
    cbar_kwargs={"label": "Mean Cloud Cover"}
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
#gl = ax.gridlines(draw_labels=True)
#gl.xlocator = mticker.FixedLocator(range(-90, -39, 10))
#gl.ylocator = mticker.FixedLocator(range(-10, 11, 10))

ax.set_title(f"Mean {header} 2002–2026")
plt.savefig(f'/home/oscholz/plots/means/Mean_{data}.png')
plt.tight_layout()
plt.show()

#---------------------------------------------------------------------------------------
#pvalue
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.PlateCarree()})

(trend_modis["pvalue"]).plot.pcolormesh(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    vmin=0,   
    vmax=0.1, 
    cbar_kwargs={"label": "p-wert"}
)

# Non-significant areas hatched (p > 0.05)
#sig_mask = trend_modis["pvalue"] > 0.05
#ax.contourf(
#    trend_modis.longitude, trend_modis.latitude,
#    sig_mask.values,
#    levels=[0.5, 1.5],
#    hatches=["////"],
#    colors="none",
#    transform=ccrs.PlateCarree()
#)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True)
gl.xlocator = mticker.FixedLocator(range(-90, -39, 10))
gl.ylocator = mticker.FixedLocator(range(-10, 11, 10))

ax.set_title(f"Trend {header} 2002–2026\n(Hatching = not significant, p > 0.05)")
#plt.savefig(f'/home/oscholz/plots/{folder}/Trend_{data}.png')
plt.tight_layout()
plt.show()

#--------------------------------------------------------------------------------------
#trend

fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={"projection": ccrs.PlateCarree()})

(trend_modis["trend_Mean"]).plot.pcolormesh(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    vmin=cmin,   
    vmax=cmax, 
    cbar_kwargs={"label": "Trend (per decade)"}
)

# Non-significant areas hatched (p > 0.05)
sig_mask = trend_modis["pvalue"] > 0.05
ax.contourf(
    trend_modis.longitude, trend_modis.latitude,
    sig_mask.values,
    levels=[0.5, 1.5],
    hatches=["////"],
    colors="none",
    transform=ccrs.PlateCarree()
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

gl = ax.gridlines(draw_labels=True)
gl.xlocator = mticker.FixedLocator(range(-90, -39, 10))
gl.ylocator = mticker.FixedLocator(range(-10, 11, 10))

ax.set_title(f"Trend {header} {start} – {end}\n(Hatching = not significant, p > 0.05)")
plt.savefig(f'/home/oscholz/plots/{folder}/Trend_{data}.png')
plt.tight_layout()
plt.show()

# DOD

In [ ]:
# ── 5. plot map ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(
    figsize=(12, 9),
    subplot_kw={"projection": ccrs.PlateCarree()}
)
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS,   linewidth=0.5)

im = ax.pcolormesh(
    x_mids, y_mids, combo,
    cmap=cmap_bivar, vmin=0, vmax=11,
    transform=ccrs.PlateCarree(), zorder=1
)

# Mask for NaN (not significant or no forrest)
nan_mask = np.where(np.isnan(combo), 0.4, np.nan)
ax.pcolormesh(
    x_mids, y_mids, nan_mask,
    cmap=ListedColormap(["white"]), vmin=0, vmax=1,
    transform=ccrs.PlateCarree(), zorder=1
)

ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5)
ax.set_title("Bivariate map: \n"
             f"Degree of Deforestation (DoD) & {header} Trend (low clouds)\n"
             "(only significant trends, p < 0.05)", fontsize=13)

# ── 6. Bivariate legend  ──────────────────────────────────────
legend_ax = fig.add_axes([0.02, 0.08, 0.16, 0.12])
legend_grid = np.arange(12).reshape(3, 4)
legend_ax.imshow(legend_grid, cmap=cmap_bivar, vmin=0, vmax=11,
                 origin="lower", aspect="auto")
legend_ax.set_xticks([0, 1, 2, 3])
legend_ax.set_xticklabels(["↓↓ Trend", "↓ Trend", "↑ Trend", "↑↑ Trend"], fontsize=8)
legend_ax.set_yticks([0, 1, 2])
legend_ax.set_yticklabels(["DoD\nniedrig", "mittel", "hoch"], fontsize=10)
legend_ax.set_title("Legende", fontsize=10)
legend_ax.tick_params(length=0)

plt.tight_layout()
plt.savefig(f'/home/oscholz/plots/{folder}/bivariate_map_DoD_{data}.png')
plt.show()